# ML-09: Validation and Research Claim Audit

Week 6. First I read two findings from FlyRank's March 2026 research paper (`docs/flyrank-seo-research-march-2026.pdf`) the way I'd want my own work read: where does the label come from, and does the validation design carry the claim? Then I turn the same two questions on my week 5 model: re-run it under stricter splits, audit the features for leakage, open the sealed June month once, and rewrite my own claims where they go further than the evidence.

The paper says plainly that its ML pages are exploratory and that its headline findings are observational. The questions below are the next level of rigor it points toward, not a grade.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
pd.set_option("display.max_colwidth", 80)

def find_repo_root(start: Path) -> Path:
    for folder in [start, *start.parents]:
        if (folder / "work" / "scripts" / "warehouse.py").exists():
            return folder
    raise FileNotFoundError("Run this from inside a clone of the repo.")

ROOT = find_repo_root(Path.cwd().resolve())
sys.path.insert(0, str(ROOT / "work" / "scripts"))
import warehouse as W  # shared loader + Lane 2 frame builder (weeks 3-6)

con = W.connect()

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import GroupKFold, train_test_split

SEED = 42

## 1. Two paper findings + my methodology questions

**Finding #4, "The Freshness Multiplier".** The paper reports that 365+ day pages refreshed within the last 30 days show 3.2x the health score (10.7 to 34.5) and 57x the impressions (71 to 4,039) of similarly old pages that weren't, and calls refresh timing "one of the strongest measured levers available".

*Where does the label come from?* "Refreshed" comes from days since the last update, and the comparison is between two different groups of pages at one moment. Pages don't get refreshed at random: if editors or an optimization workflow pick pages that already have visibility (the paper's own Myth #3 notes that flags can only fire on pages with enough impressions), the refreshed group starts ahead before any edit happens. I also found in week 3 that in the warehouse release the update date is a snapshot taken at export: 79.6% of my April pages carried an update date later than the decision date. If the paper's freshness field works the same way, some "refreshed" pages were updated after part of the 90-day window it's compared against, so the edit and the performance aren't in a clean order.

*Does the validation design carry the claim?* It carries "refreshed old pages and visible old pages overlap a lot" well. It doesn't yet carry "refreshing causes a 57x lift". **My question:** for the same pages, what did impressions look like in the 30 days before each refresh compared with the 30 days after, set against untouched pages of the same age, client and starting visibility? If the lift survives that comparison, the playbook's top recommendation gets much stronger.

**ML appendix, "What Predicts Growth?".** A logistic regression separates growing from declining pages with 71% holdout accuracy, using features that include days visible, impressions and content age, on an 80/20 split.

*Where does the label come from?* Growth vs decline is the trend direction: last 30 days vs the previous 30 days of impressions. Features like days visible and impressions are 90-day aggregates whose window contains both of those 30-day windows. In week 2 I showed that the starter release's version of this label can be rebuilt from the current-window columns on 100% of rows, so some of the 71% may be the model reading the outcome's own window rather than predicting it.

*Does the validation design carry the claim?* An 80/20 split over pages lets pages from the same brand sit on both sides, and with 57 brands, brand-wide movements can be learned from training rows and credited on test rows. The paper also doesn't print a base rate next to the 71%. **My question:** what does "always predict growth" score on the same holdout, and does the 71% hold on brands the model never trained on? The cell below estimates the first part from the paper's own Finding #1 counts, and section 2 measures the second effect on my own model.

In [2]:
# Majority-class floor implied by Finding #1's cohort counts (74,187 rising vs 45,272 falling pages).
rising, falling = 74_187, 45_272
majority = rising / (rising + falling)
print(f"'always predict growth' accuracy on that population: {majority:.1%}")
print(f"skill over that floor at 71% accuracy: {0.71 - majority:+.1%} points")
print("caveat: the ML sample (61.8K active pages) may have a different mix; the paper doesn't print it")

'always predict growth' accuracy on that population: 62.1%
skill over that floor at 71% accuracy: +8.9% points
caveat: the ML sample (61.8K active pages) may have a different mix; the paper doesn't print it


If the ML sample has a similar mix, 71% is about 9 points over a model that never looks at a feature. That's a real signal, and it should sit next to the 71% so a reader can see it.

## 2. My model under an honest split (before/after)

The week 5 random forest (same eight features, same settings, seed 42), evaluated three ways. Every row also scores the frozen week 4 rule on exactly the same test pages, so the comparison never moves.

- **Before:** a random 80/20 split over pages of the April decision frame, the design the paper used.
- **After, grouped:** 5-fold cross-validation grouped by client (week 5's design).
- **After, time-forward:** train on the March and April decision frames, test on the May frame (features from April, label from May). That's how the queue would be used: fit on the past, rank next month. The rows from clients that never appear in training are scored separately.

In [3]:
MONTHS = [("2026-02", "2026-03"), ("2026-03", "2026-04"), ("2026-04", "2026-05"), ("2026-05", "2026-06")]
frames = {}
for feature_month, label_month in MONTHS:
    d = W.build_frame(con, feature_month, label_month)
    d["tier"] = pd.cut(d["avg_position"], [0, 3, 10, 20, 50, np.inf], labels=False, include_lowest=True)
    d["client_momentum"] = d.groupby("client_hash_id")["momentum"].transform("median")
    frames[label_month] = d
pd.DataFrame({m: {"pages": len(d), "clients": d["client_hash_id"].nunique(), "base rate": round(d["label"].mean(), 3)}
              for m, d in frames.items()}).T.rename_axis("label month")

,pages,clients,base rate
label month,,,
2026-03,57656.0,24.0,0.257
2026-04,84347.0,36.0,0.270
2026-05,98956.0,41.0,0.289
2026-06,105270.0,40.0,0.263


In [4]:
WEEK5_FEATURES = W.FIVE_FEATURES + ["ctr_pct", "ctr_gap", "client_momentum"]

def add_ctr_gap(train, test):
    med = train.groupby("tier")["ctr_pct"].median()
    for d in (train, test):
        d["ctr_gap"] = d["ctr_pct"] - d["tier"].map(med)

def frozen_baseline(d):
    med = d.groupby("tier")["ctr_pct"].transform("median")
    flagged = d["momentum"].lt(-0.1) & (d["ctr_pct"] - med).lt(0)
    return np.where(flagged, -d["momentum"], 0.0)

def forest():
    return RandomForestClassifier(n_estimators=300, min_samples_leaf=50, n_jobs=-1, random_state=SEED)

def metrics(y, s, lost):
    y, s = np.asarray(y), np.asarray(s)
    return {"ROC AUC": roc_auc_score(y, s), "avg precision": average_precision_score(y, s),
            "P@50": W.precision_at_k(s, y, 50), "P@500": W.precision_at_k(s, y, 500),
            "demand@50": W.demand_captured_at_k(s, y, lost, 50), "base rate": y.mean(), "rows": len(y)}

def run_designs(features):
    rows = {}
    april = frames["2026-04"].copy()
    april["baseline"] = frozen_baseline(april)

    # Before: random 80/20 over pages.
    tr, te = train_test_split(np.arange(len(april)), test_size=0.2, random_state=SEED)
    A, B = april.iloc[tr].copy(), april.iloc[te].copy()
    add_ctr_gap(A, B)
    s = forest().fit(A[features], A["label"]).predict_proba(B[features])[:, 1]
    rows["before: random 80/20 | forest"] = metrics(B["label"], s, B["lost_per_day"])
    rows["before: random 80/20 | rule"] = metrics(B["label"], B["baseline"], B["lost_per_day"])

    # After: grouped by client.
    oof = np.zeros(len(april))
    for tr, te in GroupKFold(n_splits=5).split(april, april["label"], april["client_hash_id"]):
        A, B = april.iloc[tr].copy(), april.iloc[te].copy()
        add_ctr_gap(A, B)
        oof[te] = forest().fit(A[features], A["label"]).predict_proba(B[features])[:, 1]
    rows["after: grouped by client | forest"] = metrics(april["label"], oof, april["lost_per_day"])
    rows["after: grouped by client | rule"] = metrics(april["label"], april["baseline"], april["lost_per_day"])

    # After: time-forward, train Mar + Apr decisions, test May decision.
    train = pd.concat([frames["2026-03"], frames["2026-04"]]).copy()
    test = frames["2026-05"].copy()
    add_ctr_gap(train, test)
    s = forest().fit(train[features], train["label"]).predict_proba(test[features])[:, 1]
    b = frozen_baseline(test)
    rows["after: time-forward to May | forest"] = metrics(test["label"], s, test["lost_per_day"])
    rows["after: time-forward to May | rule"] = metrics(test["label"], b, test["lost_per_day"])
    unseen = ~test["client_hash_id"].isin(train["client_hash_id"]).to_numpy()
    rows["  May, unseen clients only | forest"] = metrics(test["label"][unseen], s[unseen], test["lost_per_day"][unseen])
    rows["  May, unseen clients only | rule"] = metrics(test["label"][unseen], b[unseen], test["lost_per_day"][unseen])
    out = pd.DataFrame(rows).T
    out["rows"] = out["rows"].astype(int)
    return out.round(3)

week5_designs = run_designs(WEEK5_FEATURES)
week5_designs

,ROC AUC,avg precision,P@50,P@500,demand@50,base rate,rows
before: random 80/20 | forest,0.767,0.571,0.98,0.814,0.024,0.271,16870
before: random 80/20 | rule,0.593,0.393,0.96,0.774,0.034,0.271,16870
after: grouped by client | forest,0.706,0.497,0.94,0.820,0.007,0.270,84347
after: grouped by client | rule,0.595,0.389,0.96,0.874,0.069,0.270,84347
after: time-forward to May | forest,0.732,0.580,0.94,0.928,0.013,0.289,98956
after: time-forward to May | rule,0.634,0.471,0.98,0.920,0.137,0.289,98956
"May, unseen clients only | forest",0.748,0.596,0.92,0.826,0.035,0.281,9198
"May, unseen clients only | rule",0.662,0.498,0.82,0.822,0.088,0.281,9198


Before and after, side by side.

The random split flatters the forest: ROC AUC 0.767 and average precision 0.571, against 0.706 and 0.497 once whole clients are held out. Roughly a third of the forest's apparent edge over random ranking on average precision disappears when it can't learn clients from their own pages. The rule doesn't move (0.593 vs 0.595 AUC), because it learns nothing from training data, which is exactly why the gap measures memorisation and not a harder test set.

The time-forward test is the encouraging part. Trained on March and April decisions and scored on May, the forest reaches 0.732 AUC and 0.580 average precision, and on the 7 clients it never saw in training it holds at 0.748 and 0.596. So the signal isn't a one-month accident, and it carries across clients in a later month. The rule improves in May too (0.634 AUC), which says May was an easier month rather than that the model got better; the base rate also moved, from 27.0% to 28.9%.

The top of the ranking doesn't change the week 5 picture. In every design the forest and the rule sit within a few points of each other at 50 and 500, while the forest stays ahead on the full-list metrics.

## 3. Leakage audit

The week 3 hunt, run again on the final feature set. First the checklist, as assertions that fail loudly.

In [5]:
LABEL_SIDE = {"imp_next", "change_pct", "client_median_change", "vs_client", "label", "lost_per_day"}
PRODUCT_FLAGS = {"health_score", "priority_score", "action_type", "needs_ctr_fix", "is_quick_win", "refresh_tier"}
SNAPSHOT_FIELDS = {"content_updated_date", "last_optimized_date", "word_count", "char_count", "is_deleted", "is_published"}

for m, d in frames.items():
    assert (pd.Period(d["feature_month"].iloc[0]) + 1) == pd.Period(d["label_month"].iloc[0]), "windows out of order"
assert not set(WEEK5_FEATURES) & LABEL_SIDE, "label-side column in the features"
assert not set(WEEK5_FEATURES) & SNAPSHOT_FIELDS, "July snapshot field in the features"
assert not any(set(d.columns) & PRODUCT_FLAGS for d in frames.values()), "product flag present"
print("timeline: features from month M, label from month M+1, no shared day   OK")
print("no label-derived, snapshot or product-flag column among the features    OK")
print("ctr_gap tier medians fitted on training rows only (add_ctr_gap)          OK")
print("split grouped by client or forward in time, base rate printed per row     OK")

timeline: features from month M, label from month M+1, no shared day   OK
no label-derived, snapshot or product-flag column among the features    OK
ctr_gap tier medians fitted on training rows only (add_ctr_gap)          OK
split grouped by client or forward in time, base rate printed per row     OK


**Suspect 1: `client_momentum`.** In week 5, shuffling it made held-out average precision *better*, which is how a feature that encodes "which client is this" behaves. Train-without test, under all three designs:

In [6]:
SEVEN = [f for f in WEEK5_FEATURES if f != "client_momentum"]
without = run_designs(SEVEN)
compare = pd.DataFrame({
    "AP with client_momentum": week5_designs["avg precision"],
    "AP without": without["avg precision"],
    "AUC with": week5_designs["ROC AUC"],
    "AUC without": without["ROC AUC"],
}).loc[[i for i in week5_designs.index if "forest" in i]]
compare["AP change"] = (compare["AP without"] - compare["AP with client_momentum"]).round(3)
compare

,AP with client_momentum,AP without,AUC with,AUC without,AP change
before: random 80/20 | forest,0.571,0.551,0.767,0.748,-0.020
after: grouped by client | forest,0.497,0.488,0.706,0.707,-0.009
after: time-forward to May | forest,0.580,0.583,0.732,0.729,0.003
"May, unseen clients only | forest",0.596,0.604,0.748,0.750,0.008


Removing `client_momentum` costs 0.020 average precision under the random split and 0.009 under the grouped split, and *gains* 0.003 and 0.008 on the time-forward test and its unseen clients. That's the pattern of a feature that helps most where clients repeat between train and test and least where they don't. On the honest designs every change is under 0.01, so it adds nothing I can defend. It comes out: the final model uses seven features.

**Suspect 2: staleness from the snapshot.** Week 3 excluded `content_updated_date` because it's a July snapshot. Here's what including it would have done: days since last update, computed from that snapshot at each decision date, added to the seven features, time-forward to May.

In [7]:
dates = con.sql(f"SELECT content_hash_id, content_updated_date FROM read_parquet('{W.file_path('dim_content.parquet')}')").df()
for m, d in frames.items():
    upd = d[["content_hash_id"]].merge(dates, on="content_hash_id", how="left")["content_updated_date"]
    d["days_since_update_snapshot"] = (pd.Timestamp(f"{m}-01") - pd.to_datetime(upd.to_numpy())).days

train = pd.concat([frames["2026-03"], frames["2026-04"]]).copy()
test = frames["2026-05"].copy()
add_ctr_gap(train, test)
for name, cols in [("seven features", SEVEN), ("+ snapshot staleness", SEVEN + ["days_since_update_snapshot"])]:
    s = forest().fit(train[cols], train["label"]).predict_proba(test[cols])[:, 1]
    r = metrics(test["label"], s, test["lost_per_day"])
    print(f"{name:22s} AUC {r['ROC AUC']:.3f}   AP {r['avg precision']:.3f}")
print(f"May test pages whose 'last update' is after the decision date: {(test['days_since_update_snapshot'] < 0).mean():.1%}")

seven features         AUC 0.729   AP 0.583


+ snapshot staleness   AUC 0.730   AP 0.584
May test pages whose 'last update' is after the decision date: 80.9%


Adding snapshot staleness changes almost nothing here (AP 0.583 to 0.584). The leak is real on paper, since 80.9% of the May test pages carry an update date after the decision date, but this forest found no use for it. That's not a reason to put it back. A field that's mostly in the future for the pages being scored stays excluded on design grounds, whether or not one model happens to exploit it; the next model might.

### The sealed month, opened once

Final configuration: the seven features, the random forest, trained on every earlier decision frame (March, April and May labels), tested on the June label month. Nothing about the model was tuned after this cell was first run.

In [8]:
FINAL = SEVEN
train = pd.concat([frames["2026-03"], frames["2026-04"], frames["2026-05"]]).copy()
june = frames["2026-06"].copy()
add_ctr_gap(train, june)
june["forest"] = forest().fit(train[FINAL], train["label"]).predict_proba(june[FINAL])[:, 1]
june["rule"] = frozen_baseline(june)
june["demand"] = june["imp"] / 31

rows = {}
for name, col in [("forest", "forest"), ("rule", "rule")]:
    rows[f"June | {name}"] = metrics(june["label"], june[col], june["lost_per_day"])
    rows[f"June | {name} x demand"] = metrics(june["label"], june[col] * june["demand"], june["lost_per_day"])
sealed = pd.DataFrame(rows).T
sealed["rows"] = sealed["rows"].astype(int)
sealed.round(3)

,ROC AUC,avg precision,P@50,P@500,demand@50,base rate,rows
June | forest,0.703,0.496,0.84,0.810,0.034,0.263,105270
June | forest x demand,0.518,0.297,0.48,0.430,0.604,0.263,105270
June | rule,0.602,0.399,1.00,1.000,0.085,0.263,105270
June | rule x demand,0.597,0.361,0.78,0.924,0.514,0.263,105270


In [9]:
rule_top = june.sort_values("rule", ascending=False).head(500)
top_client = rule_top["client_hash_id"].value_counts()
print(f"rule top 500: {top_client.iloc[0]} pages from one client, {top_client.size} clients in total")
c = top_client.index[0]

event = june[june["client_hash_id"] == c]
picked = rule_top[rule_top["client_hash_id"] == c]
print(f"that client: {len(event):,} pages in the June frame, median June change {event['change_pct'].median():.1f}%")
print(f"its pages in the rule's top 500: median late-May momentum {picked['momentum'].median():.2f}, "
      f"visible on fewer than 20 May days {(picked['days_visible'] < 20).mean():.0%}, "
      f"zero June impressions {(picked['imp_next'] == 0).mean():.0%}")

rest = june[june["client_hash_id"] != c]
print()
print("June without that client:")
pd.DataFrame({name: metrics(rest["label"], rest[name], rest["lost_per_day"]) for name in ["forest", "rule"]}).T.round(3)

rule top 500: 495 pages from one client, 5 clients in total
that client: 8,849 pages in the June frame, median June change -63.7%
its pages in the rule's top 500: median late-May momentum -5.74, visible on fewer than 20 May days 76%, zero June impressions 46%

June without that client:


,ROC AUC,avg precision,P@50,P@500,demand@50,base rate,rows
forest,0.696,0.476,0.82,0.814,0.031,0.263,96421.0
rule,0.593,0.372,0.82,0.804,0.087,0.263,96421.0


June, opened once, says three things.

The forest's full-list numbers held: 0.703 AUC and 0.496 average precision, almost exactly the grouped April result (0.706 and 0.497). That's the most reassuring number in the notebook.

The rule's perfect 1.00 at 50 and at 500 is not what it looks like. 495 of its top 500 pages come from a single client. Most of those picked pages had dropped out of search in the second half of May (median momentum -5.7, three quarters visible on fewer than 20 May days), and 46% had no June impressions at all. A precision of 1.00 there is one client's churn counted 495 times. Without that client, the rule and the forest tie at 0.82 for the top 50 and are a point apart at 500, which matches every other test.

Demand weighting repeated its week 5 trade-off: forest x demand recovers 0.604 of the possible lost demand in its top 50 against 0.514 for rule x demand, with precision 0.48 against 0.78.

### Failure examples from the sealed month

In [10]:
show = ["content_hash_id", "forest", "rule", "imp", "days_visible", "avg_position", "momentum", "ctr_pct",
        "change_pct", "client_median_change", "label"]
ranked = rest.sort_values("forest", ascending=False).reset_index(drop=True)
print("Highest-ranked pages that did NOT decline (forest, June, excluding the event client):")
ranked[ranked["label"] == 0][show].head(3).round(3)

Highest-ranked pages that did NOT decline (forest, June, excluding the event client):


,content_hash_id,forest,rule,imp,days_visible,avg_position,momentum,ctr_pct,change_pct,client_median_change,label
0,content_b3df4c79f98d291b,0.959,2.975,4308.0,30,0.380,-2.975,0.0,-44.543,-36.331,0
4,content_562c8dd6243a469d,0.931,2.551,4436.0,31,1.486,-2.551,0.0,-23.735,-36.331,0
17,content_8102fa19bf48daac,0.908,1.384,244.0,23,2.012,-1.384,0.0,64.317,-36.331,0


In [11]:
print("Biggest real declines the forest ranked outside its top 2,000 (June, excluding the event client):")
missed = ranked.iloc[2000:]
missed[missed["label"] == 1].sort_values("lost_per_day", ascending=False)[show + ["lost_per_day"]].head(3).round(3)

Biggest real declines the forest ranked outside its top 2,000 (June, excluding the event client):


,content_hash_id,forest,rule,imp,days_visible,avg_position,momentum,ctr_pct,change_pct,client_median_change,label,lost_per_day
6906,content_9648c4d1595a0794,0.490,0.000,219982.0,31,1.946,0.755,0.024,-97.181,-41.784,1,6896.160
2965,content_39e19a3ec2d95f9d,0.612,0.923,189298.0,31,10.080,-0.923,0.001,-96.301,13.175,1,5880.487
13569,content_65c75874a23fca87,0.375,0.000,191566.0,31,8.697,2.629,0.017,-87.610,-47.845,1,5413.915


Two failure patterns, both already seen in April, now confirmed on the sealed month.

**Top-ranked pages that didn't decline sit on the label's edge.** The first two fell hard in late May (momentum around -2.5 to -3, zero CTR, and the impossible below-1 positions again) and did fall in June, by 45% and 24%. But their client fell 36%, so neither was 20 points worse than its site, and the label says 0. The third dropped in late May and then grew 64% in June. One kind of error sits on a threshold I chose; the other is a temporary wobble that one month of features can't see.

**The biggest misses are large, steady pages that collapsed with no warning.** Three pages with 189K to 220K May impressions, visible every day, two of them rising in late May, lost 88% to 97% in June. One did it while its client grew 13%, so it's a page-specific event. Nothing in a one-month window anticipates that; it's the clearest case for testing longer feature windows and position-change features in the capstone.

## 4. Claim rewrite

My boldest sentences so far, and what the evidence actually supports.

In [12]:
claims = pd.DataFrame([
    ("Week 4", "At the very top it's close to perfect: 20 of the first 20 and 48 of the first 50 declined.",
     "On the April decision month, 48 of the rule's first 50 pages were measured as declining (base rate 27%); "
     "about half of those had already dropped out of search in late March, so the rule mostly surfaces "
     "pages that have already gone, which is decision-support for an indexing check rather than early warning."),
    ("Week 5", "Weighted by March demand, the forest captures 0.508 of it at 50 against the rule's 0.290.",
     "In one month of client-grouped cross-validation, demand-weighting the forest's score recovered more of the "
     "observed lost impressions in its top 50 than demand-weighting the rule did, at lower precision; on the "
     "sealed June month the same pattern held (0.60 vs 0.51 of lost demand at 50, with precision 0.48 vs 0.78), "
     "so the trade-off is measured twice but on one release and one label definition."),
    ("Week 5", "The models rank the whole list better.",
     "Across grouped, time-forward and sealed-month tests the forest's ROC AUC and average precision were measured "
     "above the rule's every time, while at the top of the list (50 to 500 pages) the two stayed within about five "
     "points of each other; all of it comes from one release, 24 to 41 clients per month and a client-relative "
     "label I defined myself."),
], columns=["where", "original", "rewrite"])
pd.set_option("display.max_colwidth", None)
claims

,where,original,rewrite
0,Week 4,At the very top it's close to perfect: 20 of the first 20 and 48 of the first 50 declined.,"On the April decision month, 48 of the rule's first 50 pages were measured as declining (base rate 27%); about half of those had already dropped out of search in late March, so the rule mostly surfaces pages that have already gone, which is decision-support for an indexing check rather than early warning."
1,Week 5,"Weighted by March demand, the forest captures 0.508 of it at 50 against the rule's 0.290.","In one month of client-grouped cross-validation, demand-weighting the forest's score recovered more of the observed lost impressions in its top 50 than demand-weighting the rule did, at lower precision; on the sealed June month the same pattern held (0.60 vs 0.51 of lost demand at 50, with precision 0.48 vs 0.78), so the trade-off is measured twice but on one release and one label definition."
2,Week 5,The models rank the whole list better.,"Across grouped, time-forward and sealed-month tests the forest's ROC AUC and average precision were measured above the rule's every time, while at the top of the list (50 to 500 pages) the two stayed within about five points of each other; all of it comes from one release, 24 to 41 clients per month and a client-relative label I defined myself."


The pattern across all three rewrites is the same. I'd been describing the month I measured as if it were the method's general behaviour, and calling the rule's top "near perfect" when much of it is pages that had already gone. The rewritten versions keep the numbers, name the scope (one release, one label I defined, 24 to 41 clients a month), and say what kind of decision each result supports.

The sentence I'd now put in the capstone abstract: on four monthly decisions from the 2026 warehouse release, a seven-feature random forest was measured ahead of a transparent rule on full-list ranking quality under client-grouped and time-forward validation, while the two were roughly level for the top 50 to 500 pages a team could actually review; demand-weighting the forest's score pointed reviewers toward larger observed declines at a cost in precision. That's decision-support evidence, not a claim that either method prevents a decline.

## Self-check

- [x] Every section above is filled, markdown thinking and the code that backs it
- [x] The notebook runs top to bottom with no errors (seed 42 throughout; June opened once)
- [x] No client names, URLs, or private queries anywhere (pseudonymized hashes and aggregates only)
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`